## Back-propagation closure rule

The current back-propagation rules only reset the direct opposite contributors of a quality. The missing rule is to reset every already-satisfied element whose decomposition chain has the same eventual contribution type to the affected quality. For goals and tasks, the reset value is pending; for qualities, the reset value is unknown (`?`) to avoid keeping a conflicting quality judgment.

```latex
\[
\Opp(\Make)=\Break,
\qquad
\Opp(\Break)=\Make .
\]

\[
\Reach(e)
:=\text{the elements reachable from }e\text{ by repeatedly following}
\]
\[
\text{(i) parent links in }\Par\text{ and (ii) dependency-transfer links from dependee side to dependum/depender side.}
\]

\[
\ECSet_v(q)
:=
\left\{
e\in \GSet(\Gmodel)\cup\Tset(\Gmodel)\cup\Qset(\Gmodel)
\middle|
\exists p\in\Reach(e)\cup\{e\}:\Clink(p,q)=v
\right\},
\qquad v\in\{\Make,\Break\}.
\]

\[
\infer
{
  \Gmodel \vdash \GMmarking \xrightarrow{\name(q)} \GMmarking'
}
{
  \begin{array}{c}
  c\in\GSet(\Gmodel)\cup\Tset(\Gmodel),\quad q\in\Qset(\Gmodel) \\
  \GMmarking(q)=sv_{\Opp(v)},\quad \GMmarking(c)=\satisfied,\quad \Clink(c,q)=v
  \end{array}
}
\quad
\GMmarking'
=
\GMmarking\left[
q\mapsto sv_v,
\ e\mapsto\Reset(e)\quad
\forall e\in\ECSet_{\Opp(v)}(q):\GMmarking(e)=\satisfied
\right]
\]

where \(sv_{\Make}=\satisfied\), \(sv_{\Break}=\denied\), and
\[
\Reset(e)=
\begin{cases}
\unknown, & e\in\Qset(\Gmodel),\\
\pending, & e\in\GSet(\Gmodel)\cup\Tset(\Gmodel).
\end{cases}
\]
```

Intuitively: when a Make contribution repairs a denied quality, all satisfied elements with eventual Break contribution to that quality are reset. Symmetrically, when a Break contribution denies a satisfied quality, all satisfied elements with eventual Make contribution to that quality are reset. Goals and tasks are reset to pending; qualities are reset to unknown (`?`).

In [1]:
from Semantics.enums import ElementStatus, LinkType
from Semantics.goal_model import GoalModel
from IPython.display import HTML, clear_output, display
from Ui.interface import InterfaceBuilder
import ipywidgets as widgets
import matplotlib.pyplot as plt
from io import BytesIO


def _has_eventual_contribution_type(self, element: str, quality: str, contribution_type: LinkType) -> bool:
    pending = [element]
    seen: set[str] = set()
    while pending:
        current = pending.pop()
        if current in seen:
            continue
        seen.add(current)
        if self._contribution_value(current, quality) == contribution_type:
            return True
        pending.extend(self._parents(current) - seen)
        pending.extend({
            child
            for parent, child, link_type in self.links
            if parent == current and link_type == LinkType.DEPENDENCY and child not in seen
        })
    return False


def _reset_qualities_supported_by(self, element: str, *, excluding: str) -> set[str]:
    impacted_qualities: set[str] = set()
    for dependent_quality in sorted(self.qualities):
        if dependent_quality == excluding:
            continue
        has_contribution = (
            self._has_eventual_contribution_type(element, dependent_quality, LinkType.MAKE)
            or self._has_eventual_contribution_type(element, dependent_quality, LinkType.BREAK)
        )
        if has_contribution and self.get_quality_status(dependent_quality) != ElementStatus.UNKNOWN:
            self.set_quality_status(dependent_quality, ElementStatus.UNKNOWN)
            impacted_qualities.add(dependent_quality)
    return impacted_qualities


def _reset_eventual_contributors(self, quality: str, contribution_type: LinkType) -> set[str]:
    impacted_elements: set[str] = set()
    for element in sorted(set(self.tasks) | set(self.goals) | set(self.qualities)):
        if not self._has_eventual_contribution_type(element, quality, contribution_type):
            continue
        if element in self.qualities and self.get_quality_status(element) == ElementStatus.SATISFIED:
            self.set_quality_status(element, ElementStatus.UNKNOWN)
            impacted_elements.add(element)
        elif element not in self.qualities and self.get_element_status(element) == ElementStatus.SATISFIED:
            self.set_element_status(element, ElementStatus.PENDING)
            impacted_elements.add(element)
            impacted_elements.update(self._reset_qualities_supported_by(element, excluding=quality))
    return impacted_elements


def try_bpfulfill_rule_with_eventual_closure(self, quality: str) -> bool:
    make_links = [link for link in self.links if link[0] == quality and link[2] == LinkType.MAKE]
    if not any(self.get_element_status(link[1]) == ElementStatus.SATISFIED for link in make_links):
        return False
    if self.get_quality_status(quality) != ElementStatus.DENIED:
        return False

    self.set_quality_status(quality, ElementStatus.SATISFIED)
    impacted_elements = self._reset_eventual_contributors(quality, LinkType.BREAK)
    if impacted_elements:
        self._propagate_break_dependency_effects(impacted_elements)
    return True


def try_bpdeny_rule_with_eventual_closure(self, quality: str) -> bool:
    break_links = [link for link in self.links if link[0] == quality and link[2] == LinkType.BREAK]
    if not any(self.get_element_status(link[1]) == ElementStatus.SATISFIED for link in break_links):
        return False
    if self.get_quality_status(quality) != ElementStatus.SATISFIED:
        return False

    self.set_quality_status(quality, ElementStatus.DENIED)
    impacted_elements = self._reset_eventual_contributors(quality, LinkType.MAKE)
    if impacted_elements:
        self._propagate_break_dependency_effects(impacted_elements)
    return True


def _apply_contribution_closure_from(self, element: str) -> bool:
    changed = False
    for quality, child, link_type in list(self.links):
        if child != element or link_type not in {LinkType.MAKE, LinkType.BREAK}:
            continue

        if link_type == LinkType.MAKE:
            if self.get_quality_status(quality) != ElementStatus.SATISFIED:
                self.set_quality_status(quality, ElementStatus.SATISFIED)
                changed = True
            impacted_elements = self._reset_eventual_contributors(quality, LinkType.BREAK)
        else:
            if self.get_quality_status(quality) != ElementStatus.DENIED:
                self.set_quality_status(quality, ElementStatus.DENIED)
                changed = True
            impacted_elements = self._reset_eventual_contributors(quality, LinkType.MAKE)

        if impacted_elements:
            changed = True
            self._propagate_break_dependency_effects(impacted_elements)
    return changed


def fire_element_with_repeated_contribution(self, element: str) -> None:
    self.changed_elements.clear()
    was_already_satisfied = (
        element in self.tasks or element in self.goals
    ) and self.get_element_status(element) == ElementStatus.SATISFIED

    self.fire_elements({element})

    if was_already_satisfied:
        if self._apply_contribution_closure_from(element):
            self.changed_elements.add(element)
        dep_children = {
            link[1]
            for link in self.links
            if link[0] == element and link[2] == LinkType.DEPENDENCY
        }
        self.fire_elements(self._parents(element) | dep_children)


def _render_trace_html(self) -> str:
    trace_html = """
    <div style='margin-top: 15px; padding: 10px; border: 1px solid #ccc; border-radius: 3px; background-color: white; font-size: 11px;'>
        <div style='font-weight: bold; margin-bottom: 8px; color: #2E86AB;'>Execution Trace</div>
    """
    if not self.executed_events:
        trace_html += "<div style='color: #666; font-style: italic;'>No events executed</div>"
    else:
        trace_html += "<div style='word-wrap: break-word;'>"
        trace_html += "<span style='color: #666;'>trace &lang;</span>"
        for i, event in enumerate(self.executed_events):
            if i > 0:
                trace_html += "<span style='color: #666;'>, </span>"
            trace_html += f"<span style='color: #2E86AB; font-weight: bold;'>{event}</span>"
        trace_html += "<span style='color: #666;'>&rang;</span></div>"
    trace_html += "</div>"
    return trace_html


def update_trace_single_slot(self):
    self.trace_output.value = self._render_trace_html()


def safe_update_visualization_single_slot(self):
    if self._update_state['updating']:
        self._update_state['pending_update'] = True
        return

    self._update_state['updating'] = True
    self._update_state['pending_update'] = False
    try:
        fig = plt.figure(figsize=(18, 16))
        if not self.whatif:
            gs = fig.add_gridspec(3, 1, height_ratios=[1.2, 1.2, 0.4], hspace=0.35)
            ax1 = fig.add_subplot(gs[0, 0])
            ax2 = fig.add_subplot(gs[1, 0])
            ax3 = fig.add_subplot(gs[2, 0])
            self._draw_petri_net(ax1)
            self._draw_goal_model(ax2)
            self._draw_mapping_table(ax3)
        else:
            gs = fig.add_gridspec(1, 1, height_ratios=[1], hspace=0.35)
            ax2 = fig.add_subplot(gs[0, 0])
            self._draw_goal_model(ax2)
        plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05, hspace=0.35)
        buffer = BytesIO()
        fig.savefig(buffer, format='png', bbox_inches='tight', dpi=120)
        plt.close(fig)
        self.viz_output.value = buffer.getvalue()
    finally:
        self._update_state['updating'] = False
        if self._update_state['pending_update']:
            self._update_state['pending_update'] = False
            self.safe_update_visualization()


if not hasattr(InterfaceBuilder, '_goccva_original_init'):
    InterfaceBuilder._goccva_original_init = InterfaceBuilder.__init__
    InterfaceBuilder._goccva_original_update_trace = InterfaceBuilder.update_trace
    InterfaceBuilder._goccva_original_safe_update_visualization = InterfaceBuilder.safe_update_visualization


def interface_init_single_slot(self, *args, **kwargs):
    previous_update_trace = InterfaceBuilder.update_trace
    previous_safe_update = InterfaceBuilder.safe_update_visualization
    InterfaceBuilder.update_trace = InterfaceBuilder._goccva_original_update_trace
    InterfaceBuilder.safe_update_visualization = InterfaceBuilder._goccva_original_safe_update_visualization
    try:
        InterfaceBuilder._goccva_original_init(self, *args, **kwargs)
    finally:
        InterfaceBuilder.update_trace = previous_update_trace
        InterfaceBuilder.safe_update_visualization = previous_safe_update

    self.trace_output = widgets.HTML(
        value='',
        layout=widgets.Layout(width='100%', margin='10px 0px'),
    )
    self.viz_output = widgets.Image(
        value=b'',
        format='png',
        layout=widgets.Layout(width='100%'),
    )

    self.controls_panel.children = tuple(
        self.trace_output if child is not None and child.__class__.__name__ == 'Output' and child is not self.status_output else child
        for child in self.controls_panel.children
    )
    # The trace output is the last child in the controls panel; assign directly to avoid replacing debug/status outputs.
    controls_children = list(self.controls_panel.children)
    controls_children[-1] = self.trace_output
    self.controls_panel.children = tuple(controls_children)

    self.content_area.children = (self.viz_output,)
    self.update_trace()
    self.safe_update_visualization()


GoalModel._has_eventual_contribution_type = _has_eventual_contribution_type
GoalModel._reset_qualities_supported_by = _reset_qualities_supported_by
GoalModel._reset_eventual_contributors = _reset_eventual_contributors
GoalModel._apply_contribution_closure_from = _apply_contribution_closure_from
GoalModel.try_bpfulfill_rule = try_bpfulfill_rule_with_eventual_closure
GoalModel.try_bpdeny_rule = try_bpdeny_rule_with_eventual_closure
GoalModel.fire_element = fire_element_with_repeated_contribution
InterfaceBuilder.__init__ = interface_init_single_slot
InterfaceBuilder._render_trace_html = _render_trace_html
InterfaceBuilder.update_trace = update_trace_single_slot
InterfaceBuilder.safe_update_visualization = safe_update_visualization_single_slot

print("Back-propagation patched: opposite eventual contributors are reset; goals/tasks -> pending, qualities -> unknown.")


Back-propagation patched: opposite eventual contributors are reset; goals/tasks -> pending, qualities -> unknown.


In [2]:
from Semantics.istar_processor import read_istar_model

goal_model_path = "content/test/test_no_dependency.txt"

def run_no_dependency_sequence(sequence):
    goal_model = read_istar_model(str(goal_model_path))
    for event in sequence:
        goal_model.fire_element(event)
    print(f"After {', '.join(sequence)}")
    for name, status in goal_model.get_markings().items():
        print(f"  {name}: {status.value}")
    print()
    return goal_model

employee_break_model = run_no_dependency_sequence(["Submit Declaration", "Break"])
assert employee_break_model.get_quality_status("Increase employee satisfaction") == ElementStatus.DENIED
assert employee_break_model.get_quality_status("adequate declaration handling") == ElementStatus.UNKNOWN

admin_break_model = run_no_dependency_sequence(["Submit Declaration", "Break by Admin"])
assert admin_break_model.get_quality_status("Increase employee satisfaction") == ElementStatus.UNKNOWN
assert admin_break_model.get_quality_status("adequate declaration handling") == ElementStatus.DENIED
print("No-dependency smoke tests passed.")


After Submit Declaration, Break
  Submit Declaration: pending
  Break: satisfied
  Break by Admin: unknown
  Money reimbursed: pending
  Money Reimbursed (Dependum): satisfied
  Money Reimbursed by Admin: satisfied
  Transaction Finished: satisfied
  Increase employee satisfaction: denied
  adequate declaration handling: unknown

After Submit Declaration, Break by Admin
  Submit Declaration: pending
  Break: unknown
  Break by Admin: satisfied
  Money reimbursed: pending
  Money Reimbursed (Dependum): pending
  Money Reimbursed by Admin: pending
  Transaction Finished: pending
  Increase employee satisfaction: unknown
  adequate declaration handling: denied

No-dependency smoke tests passed.


In [3]:
goal_model_path = "content/test/test0e_fail.txt"
goal_model = read_istar_model(str(goal_model_path), qualified=True)

goal_model.fire_element("(Admin) Break")
goal_model.fire_element("(Employee) Break")

print("After (Admin) Break, (Employee) Break")
for name, status in goal_model.get_markings().items():
    print(f"  {name}: {status.value}")

assert goal_model.get_quality_status("(Admin) adequate declaration handling") == ElementStatus.DENIED
assert goal_model.get_quality_status("(Employee) Increase employee satisfaction") == ElementStatus.DENIED

goal_model.fire_element("(Employee) Submit Declaration")
goal_model.fire_element("(Admin) Break")

print("\nAfter (Employee) Submit Declaration, repeated (Admin) Break")
for name, status in goal_model.get_markings().items():
    print(f"  {name}: {status.value}")

assert goal_model.get_element_status("(Employee) Submit Declaration") == ElementStatus.PENDING
assert goal_model.get_quality_status("(Employee) Increase employee satisfaction") == ElementStatus.UNKNOWN
assert goal_model.get_quality_status("(Admin) adequate declaration handling") == ElementStatus.DENIED
print("\nQualified dependency smoke test passed.")


After (Admin) Break, (Employee) Break
  (Employee) Submit Declaration: unknown
  (Employee) Break: satisfied
  (Admin) Break: satisfied
  (Employee) Money reimbursed: unknown
  (Admin) Money Reimbursed: unknown
  (Admin) Transaction Finished: unknown
  Money Reimbursed: unknown
  (Employee) Increase employee satisfaction: denied
  (Admin) adequate declaration handling: denied

After (Employee) Submit Declaration, repeated (Admin) Break
  (Employee) Submit Declaration: pending
  (Employee) Break: pending
  (Admin) Break: satisfied
  (Employee) Money reimbursed: pending
  (Admin) Money Reimbursed: pending
  (Admin) Transaction Finished: pending
  Money Reimbursed: pending
  (Employee) Increase employee satisfaction: unknown
  (Admin) adequate declaration handling: denied

Qualified dependency smoke test passed.


In [4]:
import ipywidgets as widgets
from IPython.display import clear_output, display
from Ui.interface import WhatIfInterfaceBuilder

clear_output(wait=False)

SCENARIOS = {
    "No dependency": ("content/test/test_no_dependency.txt", False),
    "Qualified dependency": ("content/test/test0e_fail.txt", True),
}

scenario_dropdown = widgets.Dropdown(
    options=list(SCENARIOS),
    value="Qualified dependency",
    description="Scenario:",
    layout=widgets.Layout(width="420px"),
)

# Single replacement slot: no Output widget, no nested display calls.
scenario_slot = widgets.VBox(layout=widgets.Layout(width="100%"))
page = widgets.VBox([scenario_dropdown, scenario_slot], layout=widgets.Layout(width="100%"))


def render_scenario(change=None):
    goal_model_path, qualified = SCENARIOS[scenario_dropdown.value]
    goal_model = read_istar_model(str(goal_model_path), qualified=qualified)
    interface = WhatIfInterfaceBuilder(goal_model).create_interface()
    scenario_slot.children = (interface,)


scenario_dropdown.observe(render_scenario, names="value")
render_scenario()
display(page)
